# AlphaZero-Lite Training Template
## PyTorch + python-chess for Toy Chess Experiments

This notebook provides a minimal AlphaZero-inspired training loop for chess using PyTorch and python-chess.
Perfect for experimenting with self-play, policy networks, and value networks.

### Setup and Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install python-chess numpy pandas tqdm matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import chess
import numpy as np
from collections import deque
import random
from typing import List, Tuple
import json
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Board Encoding for Neural Network

Convert chess board to 8x8x12 tensor (12 channels for 6 piece types × 2 colors)

In [ ]:
def encode_board(board: chess.Board) -> np.ndarray:
    """
    Encode board as 8x8x12 array.
    Channels: [W_pawn, W_knight, W_bishop, W_rook, W_queen, W_king,
               B_pawn, B_knight, B_bishop, B_rook, B_queen, B_king]
    """
    piece_types = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
    encoded = np.zeros((8, 8, 12), dtype=np.float32)
    
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            row, col = divmod(square, 8)
            piece_idx = piece_types.index(piece.piece_type)
            color_offset = 0 if piece.color == chess.WHITE else 6
            encoded[row, col, piece_idx + color_offset] = 1.0
    
    return encoded

# Test encoding
board = chess.Board()
encoded = encode_board(board)
print(f'Encoded board shape: {encoded.shape}')
print(f'Sample position values: {encoded[0, 0, :]}')

### Neural Network Architecture

Simple policy + value head network

In [ ]:
class ChessNet(nn.Module):
    """Policy and value network for chess."""
    
    def __init__(self, hidden_dim: int = 128):
        super(ChessNet, self).__init__()
        # Convolutional backbone
        self.conv1 = nn.Conv2d(12, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # Policy head: predict move probability for ~4672 possible moves
        self.policy_fc1 = nn.Linear(32, 128)
        self.policy_fc2 = nn.Linear(128, 4672)  # Max number of moves in UCI notation
        
        # Value head: predict game outcome [-1, 0, 1] for [Black wins, Draw, White wins]
        self.value_fc1 = nn.Linear(32, 64)
        self.value_fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # Backbone
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        
        # Global pooling
        x = self.pool(x).view(x.size(0), -1)
        
        # Policy head
        policy = self.relu(self.policy_fc1(x))
        policy = self.policy_fc2(policy)
        policy = torch.softmax(policy, dim=1)
        
        # Value head
        value = self.relu(self.value_fc1(x))
        value = torch.tanh(self.value_fc2(value))
        
        return policy, value

# Test model
model = ChessNet().to(device)
test_input = torch.randn(2, 12, 8, 8).to(device)
policy, value = model(test_input)
print(f'Policy shape: {policy.shape}, Value shape: {value.shape}')

### Self-Play Engine

Simple self-play using random legal moves (for toy experiments)

In [ ]:
def self_play_game() -> List[Tuple[np.ndarray, float]]:
    """Play one game of self-play, return list of (board_state, result)."""
    board = chess.Board()
    trajectory = []
    move_count = 0
    max_moves = 200
    
    while not board.is_game_over() and move_count < max_moves:
        # Encode board
        state = encode_board(board)
        trajectory.append(state)
        
        # Make random legal move
        move = random.choice(list(board.legal_moves))
        board.push(move)
        move_count += 1
    
    # Determine result
    if board.is_checkmate():
        # Current player (to move) is checkmated, so they lost
        result = 1.0 if board.turn == chess.BLACK else -1.0
    elif board.is_stalemate() or board.is_insufficient_material() or board.is_seventyfive_move_rule():
        result = 0.0
    else:
        result = 0.0  # Draw by move limit
    
    # Flip result for black moves
    game_trajectory = []
    for i, state in enumerate(trajectory):
        move_is_white = (i % 2 == 0)
        move_result = result if move_is_white else -result
        game_trajectory.append((state, move_result))
    
    return game_trajectory

# Test self-play
print('Testing self-play...')
trajectory = self_play_game()
print(f'Game length: {len(trajectory)} moves')
print(f'First state shape: {trajectory[0][0].shape}, Result: {trajectory[0][1]}')

### Training Loop

Simple AlphaZero-style training with self-play and network updates

In [ ]:
class GameDataset(Dataset):
    """Dataset for game trajectories."""
    
    def __init__(self, trajectories: List[List[Tuple[np.ndarray, float]]]):
        self.samples = []
        for trajectory in trajectories:
            for state, result in trajectory:
                self.samples.append((state, result))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        state, result = self.samples[idx]
        state_tensor = torch.from_numpy(state).permute(2, 0, 1).float()
        result_tensor = torch.tensor([result]).float()
        return state_tensor, result_tensor

def train_one_epoch(model, dataloader, optimizer, device):
    """Train model for one epoch."""
    model.train()
    total_loss = 0.0
    
    for batch_idx, (states, results) in enumerate(dataloader):
        states = states.to(device)
        results = results.to(device)
        
        optimizer.zero_grad()
        
        policy, value = model(states)
        
        # Loss: value regression + entropy regularization
        value_loss = nn.MSELoss()(value, results)
        policy_entropy = -(policy * torch.log(policy + 1e-10)).sum(dim=1).mean()
        total_model_loss = value_loss - 0.01 * policy_entropy
        
        total_model_loss.backward()
        optimizer.step()
        
        total_loss += total_model_loss.item()
    
    return total_loss / len(dataloader)

print('Training functions defined.')

### Main Training Loop

Run iterations of self-play + training

In [ ]:
# Hyperparameters
num_iterations = 10  # Number of self-play + training iterations
games_per_iteration = 32  # Self-play games per iteration
batch_size = 16
epochs_per_iteration = 2
learning_rate = 1e-3

# Initialize model and optimizer
model = ChessNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
history = {'iteration': [], 'loss': [], 'avg_game_length': []}

for iteration in range(num_iterations):
    print(f'\n=== Iteration {iteration + 1}/{num_iterations} ===')
    
    # Self-play
    print(f'Running {games_per_iteration} self-play games...')
    trajectories = []
    total_moves = 0
    
    for game_idx in range(games_per_iteration):
        trajectory = self_play_game()
        trajectories.append(trajectory)
        total_moves += len(trajectory)
    
    avg_game_length = total_moves / games_per_iteration
    print(f'Average game length: {avg_game_length:.1f} moves')
    
    # Training
    print(f'Training for {epochs_per_iteration} epochs...')
    dataset = GameDataset(trajectories)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    epoch_losses = []
    for epoch in range(epochs_per_iteration):
        loss = train_one_epoch(model, dataloader, optimizer, device)
        epoch_losses.append(loss)
        print(f'  Epoch {epoch + 1}/{epochs_per_iteration} - Loss: {loss:.4f}')
    
    # Record history
    avg_loss = np.mean(epoch_losses)
    history['iteration'].append(iteration + 1)
    history['loss'].append(avg_loss)
    history['avg_game_length'].append(avg_game_length)
    
    print(f'Iteration {iteration + 1} complete - Avg Loss: {avg_loss:.4f}')

print('\n=== Training Complete ===')
print(f'Final Loss: {history["loss"][-1]:.4f}')
print(f'Final Avg Game Length: {history["avg_game_length"][-1]:.1f} moves')

### Visualization

Plot training progress

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(history['iteration'], history['loss'], marker='o', label='Training Loss')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss Over Time')
ax1.grid(True)
ax1.legend()

# Game length plot
ax2.plot(history['iteration'], history['avg_game_length'], marker='s', color='orange', label='Avg Game Length')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Average Moves per Game')
ax2.set_title('Game Length Over Time')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

print('\nTraining history:', history)

### Save Model

Save trained model for later use

In [ ]:
# Save model
torch.save(model.state_dict(), '/content/alphazero_chess_model.pth')
print('Model saved to /content/alphazero_chess_model.pth')

# Save training history
with open('/content/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Training history saved to /content/training_history.json')